# Data Cleaning
Data from 5 datasets are extracted to obtain two files for each dataset:
- sensor.csv: subjectID,sessionID,taskID,timestamp,isTurn,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z, where taskID is 0 for stance with open eyes, 1 for stance with closed eyes and 2 for walking, with isTurn set to 1 in the corresponding timestamps.
- clinical.csv: subjectID,age,gender,disease_duration,h_y,updrs_iii,postural_stability.


Common units and reference axes are:
- acceleration: g
- angular velocity: rad/s
- x axis: vertical, positive up
- y axis: medio-lateral, positive right
- z axis: antero-frontal, positive backward


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

### Common functions

In [ ]:
def check_axes(df, subject_id=1, session_id=1, task_id=0):
    subset = df[
        (df['subjectID'] == subject_id) & 
        (df['sessionID'] == session_id) & 
        (df['taskID'] == task_id)
    ].copy()
    
    print(f"X orig: {subset['acc_x'].mean():+.3f} g")
    print(f"Y orig: {subset['acc_y'].mean():+.3f} g")
    print(f"Z orig: {subset['acc_z'].mean():+.3f} g")
    print(f"Var X: {subset['acc_x'].var():.5f}")
    print(f"Var Y: {subset['acc_y'].var():.5f}")
    print(f"Var Z: {subset['acc_z'].var():.5f}")

In [ ]:
sensor_cols = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
group_cols = ["subjectID", "sessionID", "taskID"]

## FoG-STAR

In [ ]:
fog_star_sensor = pd.read_csv("data/original_data/FoG-STAR/sensor_data.csv")
fog_star_sensor= fog_star_sensor[fog_star_sensor.activity>0]
fog_star_sensor.columns

In [ ]:
#keep only the columns that are needed for the analysis and rename them for better understanding
fog_star_sensor = fog_star_sensor[["timestamp", "back_acc_x", "back_acc_y", "back_acc_z", "back_gyro_x", "back_gyro_y", "back_gyro_z","subjectID", "sessionID", "taskID", "activity", "fog"]]
fog_star_sensor.columns = ["timestamp", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z" ,"subjectID", "sessionID", "taskID", "activity", "fog"]

In [ ]:
# task 2: Stand 1min, task 3: Walk 1min, task 4: Sit-to-stand
fog_star_sensor = fog_star_sensor[fog_star_sensor.taskID.isin([1, 2, 3])].copy()

fog_star_sensor['newTaskID'] = np.nan
fog_star_sensor['isTurn'] = 0

# - Task 2 (Stand 1min) -> Task 0
fog_star_sensor.loc[fog_star_sensor['taskID'] == 2, 'newTaskID'] = 0

# - Task 1 (Sit-to-stand) -> Task 10 to verify axes
mask_turns = (fog_star_sensor['taskID'] == 1) & (fog_star_sensor['activity']==4)
fog_star_sensor.loc[mask_turns, 'newTaskID'] = 10

# - Task 3 + Activity 6/7 (turns) -> Task 2 and add a column 0/1 if turn or walk
mask = (fog_star_sensor['taskID'] == 3) & (fog_star_sensor['fog'] == 0)
mask_turns = (fog_star_sensor['taskID'] == 3) & (fog_star_sensor['activity'].isin([6, 7]))
fog_star_sensor.loc[mask, 'newTaskID'] = 2
fog_star_sensor.loc[mask_turns, 'isTurn'] = 1

fog_star_sensor = fog_star_sensor.dropna(subset=['newTaskID']).copy()
fog_star_sensor['taskID'] = fog_star_sensor['newTaskID'].astype(int)
fog_star_sensor.drop(columns=['activity', 'fog', 'newTaskID'], inplace=True, errors='ignore')

print(fog_star_sensor['taskID'].value_counts())
fog_star_sensor.describe()

In [ ]:
# remove group task+subject were all record for acc e gyro are nan
fog_star_sensor = fog_star_sensor.groupby(group_cols).filter(lambda x: not x[sensor_cols].isna().all().all())

In [ ]:
# from deg/s to rad/s
fog_star_sensor["gyro_x"] = np.deg2rad(fog_star_sensor["gyro_x"])
fog_star_sensor["gyro_y"] = np.deg2rad(fog_star_sensor["gyro_y"])
fog_star_sensor["gyro_z"] = np.deg2rad(fog_star_sensor["gyro_z"])

In [ ]:
# plot the sensor data for one subject and session
subject_id = 2
session_id = 1
task_id = 0
subject_data = fog_star_sensor[(fog_star_sensor.subjectID == subject_id) & (fog_star_sensor.sessionID == session_id) & (fog_star_sensor.taskID == task_id) ]
#subject_data["acc_x"] = subject_data["acc_x"] - subject_data["acc_x"].mean()
plt.plot(subject_data.timestamp, subject_data["acc_x"], label="Acc X")
plt.plot(subject_data.timestamp, subject_data["acc_y"], label="Acc Y")
plt.plot(subject_data.timestamp, subject_data["acc_z"], label="Acc Z")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Value")
plt.legend()

In [ ]:
check_axes(fog_star_sensor, subject_id=1, session_id=1, task_id=10)

In [ ]:
fog_star_sensor["acc_z"] = -fog_star_sensor["acc_z"]
fog_star_sensor["gyro_z"] = -fog_star_sensor["gyro_z"]

In [ ]:
# remove tug task from the dataset
fog_star_sensor = fog_star_sensor[fog_star_sensor.taskID != 10].copy()

In [ ]:
fog_star_clinical = pd.read_csv("data/original_data/FoG-STAR/clinical_data.csv")
fog_star_clinical.head()

### OMNIA-PARK

In [ ]:
folder_path = "data/original_data/OMNIA-PARK/pd"

all_data = []

for file in os.listdir(folder_path):
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df = df.rename(columns={"Hips_timestamp": "Timestamp"})
        
        df = df[["subjectID", "Timestamp", "Hips_accX", "Hips_accY", "Hips_accZ", "Hips_gyroX", "Hips_gyroY", "Hips_gyroZ", "taskID"]].copy()
        df.columns = ["subjectID", "timestamp", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z", "taskID"]

        df = df[df.taskID.isin([7, 8, 2,4])]
        task_mapping = {7: 0, 8: 1, 2: 2, 4: 10}
        df.taskID = df.taskID.map(task_mapping)
        df["sessionID"] = 1
        
        all_data.append(df)

omnia_park_sensor = pd.concat(all_data, ignore_index=True)

In [ ]:
# from deg/s to rad/s
omnia_park_sensor["gyro_x"] = np.deg2rad(omnia_park_sensor["gyro_x"])
omnia_park_sensor["gyro_y"] = np.deg2rad(omnia_park_sensor["gyro_y"])
omnia_park_sensor["gyro_z"] = np.deg2rad(omnia_park_sensor["gyro_z"])

In [ ]:
# plot the sensor data for one subject and session
subject_id = 19
session_id = 1
task_id = 2
subject_data = omnia_park_sensor[(omnia_park_sensor.subjectID == subject_id) & (omnia_park_sensor.sessionID == session_id) & (omnia_park_sensor.taskID == task_id) & (6 < omnia_park_sensor.timestamp) & (omnia_park_sensor.timestamp < 10)]
#subject_data["acc_x"] = subject_data["acc_x"] - subject_data["acc_x"].mean()
plt.plot(subject_data.timestamp, subject_data["acc_x"], label="Acc X")
plt.plot(subject_data.timestamp, subject_data["acc_y"], label="Acc Y")
plt.plot(subject_data.timestamp, subject_data["acc_z"], label="Acc Z")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Value")
plt.legend()

In [ ]:
check_axes(omnia_park_sensor, subject_id=1, session_id=1, task_id=2)

In [ ]:
omnia_park_sensor = omnia_park_sensor[omnia_park_sensor.taskID != 10].copy()

In [ ]:
# remove sub19 since incorrect correspondence of tasks and sub27 only task2
omnia_park_sensor = omnia_park_sensor[~((omnia_park_sensor.subjectID == 19) & (omnia_park_sensor.taskID == 2))].copy()
omnia_park_sensor = omnia_park_sensor[~((omnia_park_sensor.subjectID == 27) & (omnia_park_sensor.taskID == 2))].copy()

In [ ]:
omnia_park_clinical = pd.read_excel("data/original_data/OMNIA-PARK/clinical_info.xlsx")
omnia_park_clinical = omnia_park_clinical.rename(columns={"mds_updrs_3": "updrs_iii"})
omnia_park_clinical.head()

## PD-PHONE

In [ ]:
folder_path = "data/original_data/PD-PHONE/pd"

all_data = []

for file in os.listdir(folder_path):
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        subject_id = file.split('.')[0].split('_')[1].replace('pd', '')  
        
        df = pd.read_csv(file_path)
        
        df = df.iloc[:, [0, 1, 2, 3, 4, 5, -1]].copy()
        df.columns = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z", "taskID"]
        
        df = df[df.taskID == 3].copy()
        
        df["taskID"] = 0
        
        df["subjectID"] = int(subject_id)
        df["sessionID"] = 1
        
        df["timestamp"] = np.arange(len(df)) / 200.0
        all_data.append(df)

pd_phone_sensor = pd.concat(all_data, ignore_index=True)
pd_phone_sensor.columns

In [ ]:
g = 9.80665
pd_phone_sensor["acc_x"] = pd_phone_sensor["acc_x"] / g
pd_phone_sensor["acc_y"] = pd_phone_sensor["acc_y"] / g
pd_phone_sensor["acc_z"] = pd_phone_sensor["acc_z"] / g
pd_phone_sensor["isTurn"] = 0

In [ ]:
# plot the sensor data for one subject and session
subject_id = 2
session_id = 1
task_id = 0
subject_data = pd_phone_sensor[(pd_phone_sensor.subjectID == subject_id) & (pd_phone_sensor.sessionID == session_id) & (pd_phone_sensor.taskID == task_id) & (6 < pd_phone_sensor.timestamp) & (pd_phone_sensor.timestamp < 10)]
#subject_data["acc_x"] = subject_data["acc_x"] - subject_data["acc_x"].mean()
plt.plot(subject_data.timestamp, subject_data["acc_x"], label="Acc X")
plt.plot(subject_data.timestamp, subject_data["acc_y"], label="Acc Y")
plt.plot(subject_data.timestamp, subject_data["acc_z"], label="Acc Z")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Value")
plt.legend()

In [ ]:
check_axes(pd_phone_sensor, subject_id=1, session_id=1, task_id=0)

In [ ]:
# exchange axes
pd_phone_sensor['acc_x'] = -pd_phone_sensor['acc_x']
pd_phone_sensor['gyro_x'] = -pd_phone_sensor['gyro_x']

In [ ]:
pd_phone_clinical = pd.read_excel("data/original_data/PD-PHONE/clinical_info.xlsx")
pd_phone_clinical.head()

### WearPD

In [ ]:
folder_path = "data/original_data/WearPD/pd"
all_data = []

col_keep = [
    'Time', 'GeneralEvent', 'ClinicalEvent',
    'LowerBack_Acc_X', 'LowerBack_Acc_Y', 'LowerBack_Acc_Z',
    'LowerBack_Gyr_X', 'LowerBack_Gyr_Y', 'LowerBack_Gyr_Z'
]

col_map = {
    'Time': 'timestamp',
    'LowerBack_Acc_X': 'acc_x', 'LowerBack_Acc_Y': 'acc_y', 'LowerBack_Acc_Z': 'acc_z',
    'LowerBack_Gyr_X': 'gyro_x', 'LowerBack_Gyr_Y': 'gyro_y', 'LowerBack_Gyr_Z': 'gyro_z'
}

for file in os.listdir(folder_path):
    if file.endswith('balance.csv') or file.endswith('selfpace_matturn.csv') or file.endswith('tug.csv'):
        file_path = os.path.join(folder_path, file)
        
        subject_id = file.split('_')[0]
        try:
            df = pd.read_csv(file_path, usecols=col_keep)
        except ValueError:
            print(f"Missing LowerBack sensor in: {file}. File skipped.")
            continue
        
    
        df_parts = [] 
        
        if 'balance' in file:
            df0 = df[df['GeneralEvent'].str.contains('EO_FeetShoWidth', na=False)].copy()
            df0['taskID'] = 0
            df0["isTurn"] = 0
            df_parts.append(df0)
                
            # Task 1: closed eyes, feet apart
            df1 = df[df['GeneralEvent'].str.contains('EC_FeetShoWidth', na=False)].copy()
            df1['taskID'] = 1
            df1["isTurn"] = 0
            df_parts.append(df1)
                
        elif 'selfpace_mat' in file:
            df2 = df.copy()
            df2['taskID'] = 2
            df2 = df2.sort_values('Time')
            
            df2['isTurn'] = 0
            
            if 'GeneralEvent' in df2.columns:
                mask_turns = df2['GeneralEvent'].str.contains('Turn', na=False)
                df2.loc[mask_turns, 'isTurn'] = 1
                
            df_parts.append(df2)

        elif "tug" in file:
            # Task 10: TUG
            df2 = df.copy()
            df2["isTurn"] = 0
            df2['taskID'] = 10
            df_parts.append(df2)
        
        if len(df_parts) > 0:
            df_file_completo = pd.concat(df_parts, ignore_index=True)
            df_file_completo = df_file_completo.rename(columns=col_map)
            df_file_completo["subjectID"] = subject_id
            df_file_completo["sessionID"] = 1
            df_file_completo.drop(columns=['GeneralEvent', 'ClinicalEvent'], inplace=True)
            df_file_completo["timestamp"] = df_file_completo["timestamp"].apply(lambda x: float(x.split(" ")[0]))
            
            all_data.append(df_file_completo)

wearpd_sensor = pd.concat(all_data, ignore_index=True)

In [ ]:
wearpd_sensor["acc_x"] = wearpd_sensor["acc_x"] / g
wearpd_sensor["acc_y"] = wearpd_sensor["acc_y"] / g
wearpd_sensor["acc_z"] = wearpd_sensor["acc_z"] / g

In [ ]:
# plot the sensor data for one subject and session
subject_id = "nls022"
session_id = 1
task_id = 10
subject_data = wearpd_sensor[(wearpd_sensor.subjectID == subject_id) & (wearpd_sensor.sessionID == session_id) & (wearpd_sensor.taskID == task_id) & (2 < wearpd_sensor.timestamp) & (wearpd_sensor.timestamp < 15)]
#subject_data["acc_x"] = subject_data["acc_x"] - subject_data["acc_x"].mean()
plt.plot(subject_data.timestamp, subject_data["acc_x"], label="Acc X")
plt.plot(subject_data.timestamp, subject_data["acc_y"], label="Acc Y")
plt.plot(subject_data.timestamp, subject_data["acc_z"], label="Acc Z")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Value")
plt.legend()

In [ ]:
check_axes(wearpd_sensor, subject_id="nls002", session_id=1, task_id=10)

In [ ]:
wearpd_sensor['acc_x'] = -wearpd_sensor['acc_x']
wearpd_sensor['gyro_x'] = -wearpd_sensor['gyro_x']

In [ ]:
wearpd_sensor = wearpd_sensor[wearpd_sensor.taskID != 10].copy()

In [ ]:
# remove group task+subject were all record for acc e gyro are nan
wearpd_sensor = wearpd_sensor.groupby(group_cols).filter(lambda x: not x[sensor_cols].isna().all().all())

In [ ]:
wearpd_clinical = pd.read_csv("data/original_data/WearPD/PD - Demographic+Clinical - datasetV1.csv",header=[0, 1])
wearpd_clinical.columns

cols = []
for level1, level2 in wearpd_clinical.columns:
    l1 = str(level1).strip()
    l2 = str(level2).strip()
    if "Unnamed" in l1 or l1 == "-":
        cols.append(l2)
    else:
        cols.append(f"{l1}_{l2}")

wearpd_clinical.columns = cols

df_pd = wearpd_clinical.dropna(subset=['Subject ID']).copy()
df_pd['Subject ID'] = df_pd['Subject ID'].apply(lambda x: str(x).lower())

updrs3_cols = [col for col in df_pd.columns if 'MDSUPDRS_3' in col]
for col in updrs3_cols:
    df_pd[col] = pd.to_numeric(df_pd[col], errors='coerce')
df_pd['updrs_iii'] = df_pd[updrs3_cols].sum(axis=1, skipna=True)

cols_to_keep = {
    'Subject ID': 'subjectID',
    'Age (years)': 'age',
    'Sex': 'gender',                    
    'Years since PD diagnosis': 'disease_duration',
    'Modified Hoehn & Yahr Score': 'h_y',
    'updrs_iii': 'updrs_iii',        
    'POSTURAL STABILITY_MDSUPDRS_3-12': 'postural_stability'
}

wearpd_clinical = df_pd[list(cols_to_keep.keys())].rename(columns=cols_to_keep)
mapping = { "Male" : "M", "Female" : "F" }
wearpd_clinical['gender'] = wearpd_clinical['gender'].map(mapping)
wearpd_clinical.head()

## Kiel Dataset


In [ ]:
def load_kiel_mat(file_path):
    mat: dict = loadmat(file_path)
    data = mat['data']

    acc = data['acc'].item()
    gyro = data['gyro'].item()


    # MATLAB fields can be nested object arrays; unwrap until scalar.
    fs_raw = data['fs']
    while isinstance(fs_raw, np.ndarray):
        if fs_raw.size == 0:
            raise ValueError(f"Empty sampling rate in file: {file_path}")
        fs_raw = fs_raw.reshape(-1)[0]
    fs = float(fs_raw)

    locations = data['imu_location'].item()
    sensor_idx = np.where(locations == 'pelvis')[0][0]

    acc = acc[:, :, sensor_idx]
    gyro = gyro[:, :, sensor_idx]

    T = acc.shape[0]
    time = np.arange(T) / fs

    df = pd.DataFrame({
        'timestamp': time,
        'acc_x': acc[:, 0],
        'acc_y': acc[:, 1],
        'acc_z': acc[:, 2],
        'gyro_x': gyro[:, 0],
        'gyro_y': gyro[:, 1],
        'gyro_z': gyro[:, 2],
    })

    return df

In [ ]:
folder_path = "data/original_data/KielPD"

all_data = []

if not os.path.exists(folder_path):
    print(f"Error: {folder_path} not found.")
else:
    for file in os.listdir(folder_path):

        if 'imu_walk_turn' in file or 'imu_balance_sbs' in file:
            file_path = os.path.join(folder_path, file)
        else:
            continue

        subject_id = int(file.split("_")[0].replace("pp", ""))
       
        try:
            df = load_kiel_mat(file_path)
            df_parts = []

            if 'balance_sbs' in file:
                df0 = df.copy()
                df0["isTurn"] = 0
                df0['taskID'] = 0
                df_parts.append(df0)

            elif 'walk_turn' in file:
                df2 = df.copy()
                df2['taskID'] = 2
                df2['isTurn'] = 0
                df_parts.append(df2)

            if df_parts:
                df_res = pd.concat(df_parts, ignore_index=True)

                df_res["subjectID"] = subject_id

                if "on" in file:
                    df_res["sessionID"] = 1
                elif "off" in file:
                    df_res["sessionID"] = 2

                cols_fin = ['subjectID', 'sessionID', 'taskID', 'timestamp', 'isTurn','acc_x', 'acc_y', 'acc_z','gyro_x', 'gyro_y', 'gyro_z']

                df_res = df_res[[c for c in cols_fin if c in df_res.columns]]
                all_data.append(df_res)

        except Exception as e:
            print(f"Error loading {file}: {e}")

kiel_sensor = pd.concat(all_data, ignore_index=True)


In [ ]:
# plot the sensor data for one subject and session
subject_id = 8
session_id = 2
task_id = 2
subject_data = kiel_sensor[(kiel_sensor.subjectID == subject_id) & (kiel_sensor.sessionID == session_id) & (kiel_sensor.taskID == task_id) & (6 < kiel_sensor.timestamp) & (kiel_sensor.timestamp < 10)]
#subject_data["acc_x"] = subject_data["acc_x"] - subject_data["acc_x"].mean()
plt.plot(subject_data.timestamp, subject_data["acc_x"], label="Acc X")
plt.plot(subject_data.timestamp, subject_data["acc_y"], label="Acc Y")
plt.plot(subject_data.timestamp, subject_data["acc_z"], label="Acc Z")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Value")
plt.legend()

In [ ]:
# convert to rad/s
kiel_sensor["gyro_x"] = np.deg2rad(kiel_sensor["gyro_x"])
kiel_sensor["gyro_y"] = np.deg2rad(kiel_sensor["gyro_y"])
kiel_sensor["gyro_z"] = np.deg2rad(kiel_sensor["gyro_z"])

In [ ]:
kiel_sensor['acc_x'] = -kiel_sensor['acc_x']
kiel_sensor['gyro_x'] = -kiel_sensor['gyro_x']
kiel_sensor = kiel_sensor[kiel_sensor.taskID != 10].copy()

In [ ]:
# clinical data
kiel_clinical = pd.read_excel("data\\original_data\\KielPD\\PD_demographics_scores_extern.xlsx")
kiel_clinical = kiel_clinical[[
    "id", "med_state", "gender", "age", "disease_duration",
    "hoehn_u_yahr", "updrs_3_total", "updrs_3_12_postural_stab"
]]

# Keep a stable subject identifier
kiel_clinical["subjectID"] = pd.to_numeric(kiel_clinical["id"], errors="coerce").astype("Int64")
kiel_clinical = kiel_clinical.dropna(subset=["subjectID"]).copy()

# Prefer OFF when available, otherwise keep ON
kiel_clinical["med_state"] = kiel_clinical["med_state"].astype(str).str.lower().str.strip()
state_priority = {"off": 0, "on": 1}
kiel_clinical["state_priority"] = kiel_clinical["med_state"].map(state_priority).fillna(2).astype(int)

# One clinical row per subjectID with OFF priority
kiel_clinical = (
    kiel_clinical
    .sort_values(["subjectID", "state_priority"])
    .drop_duplicates(subset=["subjectID"], keep="first")
    .drop(columns=["id", "med_state", "state_priority"])
    .reset_index(drop=True)
)

kiel_clinical = kiel_clinical.rename(columns={
    "hoehn_u_yahr": "h_y",
    "updrs_3_total": "updrs_iii",
    "updrs_3_12_postural_stab": "postural_stability"
})

map_gender = {0: "M", 1: "F"}
kiel_clinical["gender"] = kiel_clinical["gender"].map(map_gender)
kiel_clinical.head()

### Ensure datasets consistency

In [ ]:
datasets = ["fog_star", "omnia_park", "pd_phone", "wearpd", "kiel"]

output_folder = "data/cleaned_data"
os.makedirs(output_folder, exist_ok=True)

TARGET_SENSOR_COLS = [
    'subjectID', 'sessionID', 'taskID', 'timestamp', 'isTurn', 
    'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z'
]

TARGET_CLINICAL_COLS = [
    'subjectID', 'age', 'gender', 'disease_duration', 
    'h_y', 'updrs_iii', 'postural_stability', 'berg', 'fes-i'
]

for dataset in datasets:
    print(f"\nDataset: {dataset}")
    
    # Assegna i dataframe
    if dataset == "fog_star":
        sensor_df = fog_star_sensor.copy()
        clinical_df = fog_star_clinical.copy()
    elif dataset == "omnia_park":
        sensor_df = omnia_park_sensor.copy()
        clinical_df = omnia_park_clinical.copy()
    elif dataset == "pd_phone":
        sensor_df = pd_phone_sensor.copy()
        clinical_df = pd_phone_clinical.copy()
    elif dataset == "wearpd":
        sensor_df = wearpd_sensor.copy()
        clinical_df = wearpd_clinical.copy()
    elif dataset == "kiel":
        sensor_df = kiel_sensor.copy()
        clinical_df = kiel_clinical.copy()
    
    subject_col = "subjectID"
    sensor_subjects = set(sensor_df[subject_col].unique())
    clinical_subjects = set(clinical_df[subject_col].unique())
    
    missing_in_clinical = sensor_subjects - clinical_subjects
    missing_in_sensor = clinical_subjects - sensor_subjects
    
    print(f"Before filtering - Sensor: {len(sensor_df)}, Clinical: {len(clinical_df)}")
    
    # FILTRA ENTRAMBI i dataframe per soggetti comuni
    common_subjects = sensor_subjects & clinical_subjects
    sensor_df = sensor_df[sensor_df[subject_col].isin(common_subjects)]
    clinical_df = clinical_df[clinical_df[subject_col].isin(common_subjects)]
    
    print(f"After filtering - Sensor: {len(sensor_df)}, Clinical: {len(clinical_df)}")
    
    if missing_in_clinical:
        print(f"  In sensor but not in clinical: {missing_in_clinical}")
    if missing_in_sensor:
        print(f"  In clinical but not in sensor: {missing_in_sensor}")
    
    df_s = sensor_df[[c for c in TARGET_SENSOR_COLS if c in sensor_df.columns]]
    for col in TARGET_SENSOR_COLS:
        if col not in df_s.columns:
            df_s[col] = np.nan
    df_s[TARGET_SENSOR_COLS].to_csv(f"{output_folder}/{dataset}_sensor.csv", index=False)
    
    # Clinical
    df_c = clinical_df[[c for c in TARGET_CLINICAL_COLS if c in clinical_df.columns]]
    for col in TARGET_CLINICAL_COLS:
        if col not in df_c.columns:
            df_c[col] = np.nan
    df_c[TARGET_CLINICAL_COLS].to_csv(f"{output_folder}/{dataset}_clinical.csv", index=False)
    
    print(f"Saved {dataset}")

In [ ]:
def check_imu_standard(group, sf):
    ref_data = group[['acc_x', 'acc_y', 'acc_z']].head(int(sf/2))
    mean_acc = ref_data.mean()
    total_norm = np.sqrt((mean_acc**2).sum())
    
    variances = group[['acc_y', 'acc_z']].var()
    
    report = {
        "is_unit_g": 0.9 <= total_norm <= 1.1,
        "is_x_vertical": mean_acc.abs().idxmax() == 'acc_x',
        "is_x_positive_up": mean_acc['acc_x'] < 0,
        "is_z_anteroposterior": variances.idxmax() == 'acc_z', # Z dovrebbe oscillare di piÃ¹
        "norm_value": round(total_norm, 3),
        "gravity_on_axis": mean_acc.idxmax()
    }
    
    if not report["is_unit_g"]:
        print(f"Unit error: {report['norm_value']}g (should be ~1g).")
    if not report["is_x_vertical"]:
        print(f"Orientation error: Gravity found on {report['gravity_on_axis']}")
    if report["is_x_vertical"] and not report["is_x_positive_up"]:
        print("Sign error: The X-axis is vertical but points downward (DOWN).")
    if not report["is_z_anteroposterior"]:
        print("Axis error: The Y-axis oscillates more than the Z-axis. Perhaps they are swapped (Antero vs Lat).")
        
    return report

In [ ]:
datasets_dict = {
    "fog_star": (fog_star_sensor, fog_star_clinical),
    "omnia_park": (omnia_park_sensor, omnia_park_clinical),
    "pd_phone": (pd_phone_sensor, pd_phone_clinical),
    "wearpd": (wearpd_sensor, wearpd_clinical),
    "kiel": (kiel_sensor, kiel_clinical)
}

for dataset_name, (df_sensor, df_clinical) in datasets_dict.items():
    print(f"\nProcessing dataset: {dataset_name}")
    
    if not df_sensor.empty:
        test_subject = df_sensor['subjectID'].iloc[0]
        test_session = df_sensor['sessionID'].iloc[0]
        test_task = df_sensor['taskID'].iloc[0]
        
        subject_data = df_sensor[(df_sensor['subjectID'] == test_subject) & (df_sensor['sessionID'] == test_session) & (df_sensor['taskID'] == test_task)]
        
        time_diff = subject_data['timestamp'].diff().mean()

        freq_hz = 1 / time_diff
        print(f"  Estimated frequency: {freq_hz:.2f} Hz")

        check_imu_standard(subject_data, freq_hz)

### Extract turns

In [ ]:
from extract_turn.extract_turn import extract_turns_from_dataset

path = f"data/cleaned_data/omnia_park_sensor.csv"
df_processed = extract_turns_from_dataset(path, sf=90, tasks_to_process=[2])

if df_processed is not None:
    output_path = f"data/cleaned_data/omnia_park_sensor.csv"
    df_processed.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

In [ ]:
path = f"data/cleaned_data/kiel_sensor.csv"
df_processed = extract_turns_from_dataset(path, sf=200, tasks_to_process=[2])
if df_processed is not None:
    output_path = f"data/cleaned_data/kiel_sensor.csv"
    df_processed.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")